# Classification Metrics

**Before running this notebook, make sure to check `00_environment_setup.md` to see whether you have set the environment correctly.**

In this notebook we will explore various classification metrics using simple examples, before moving to the basic classification guide in notebook `02_classifiers_tutorial.ipynb`.

## Outline of the notebook

> Links are clickable in Jupyter Notebook and JupyterLab — click any entry to jump to that section.

**1. [Basic Definitions](#1.-Basic-Definitions)**
- [1.1 True Positives (TP) and True Negatives (TN)](#1.1-True-Positives-(TP)-and-True-Negatives-(TN))
- [1.2 False Positives (FP) and False Negatives (FN)](#1.2-False-Positives-(FP)-and-False-Negatives-(FN))
- [1.3 Derived Rates](#1.3-Derived-Rates)
- [1.4 Classification Scenarios](#1.4-Classification-Scenarios)

**2. [Binary Classification Metrics](#2.-Binary-Classification-Metrics)**
- [2.1 Accuracy](#2.1-Accuracy)
- [2.2 Confusion Matrix](#2.2-Confusion-Matrix)
- [2.3 Precision, Recall & F1-score](#2.3-Precision,-Recall-&-F1-score)
- [2.4 Balanced Accuracy](#2.4-Balanced-Accuracy)
- [2.5 ROC AUC](#2.5-ROC-AUC)
- [2.6 PR AUC](#2.6-PR-AUC)

**3. [Multi-Class Classification Metrics](#3.-Multi-Class-Classification-Metrics)**
- [3.1 Accuracy](#3.1-Accuracy)
- [3.2 Balanced Accuracy](#3.2-Balanced-Accuracy)
- [3.3 Confusion Matrix](#3.3-Confusion-Matrix)
- [3.4 Macro, Micro & Weighted F1-score](#3.4-Macro,-Micro-&-Weighted-F1-score)
- [3.5 Interactive Multi-Class Classification Example](#3.5-Interactive-Multi-Class-Classification-Example)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
)
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Basic Definitions

We can start with some definitions for a binary classification problem.

**Running example** — we will track the following 8 predictions throughout this section:

| # | Actual (`y_true`) | Predicted (`y_pred`) |
|---|:-----------------:|:--------------------:|
| 0 | 1 (pos) | 1 (pos) |
| 1 | 0 (neg) | 0 (neg) |
| 2 | 1 (pos) | 0 (neg) |
| 3 | 1 (pos) | 1 (pos) |
| 4 | 0 (neg) | 0 (neg) |
| 5 | 1 (pos) | 1 (pos) |
| 6 | 0 (neg) | 1 (pos) |
| 7 | 0 (neg) | 0 (neg) |

In [ ]:
y_true = np.array([1, 0, 1, 1, 0, 1, 0, 0])
y_pred = np.array([1, 0, 0, 1, 0, 1, 1, 0])

print("y_true:", y_true)
print("y_pred:", y_pred)

### 1.1 True Positives (TP) and True Negatives (TN)

- **True Positives (TP)**: the model *correctly* predicts the positive class — both the actual label and the prediction are positive.

- **True Negatives (TN)**: the model *correctly* predicts the negative class — both the actual label and the prediction are negative.

In our example:  
- **TP = 3** — samples 0, 3, and 5 are positive and predicted positive.  
- **TN = 3** — samples 1, 4, and 7 are negative and predicted negative.

In [ ]:
TP = np.sum((y_true == 1) & (y_pred == 1))
TN = np.sum((y_true == 0) & (y_pred == 0))

print(f"True Positives  (TP): {TP}")
print(f"True Negatives  (TN): {TN}")

### 1.2 False Positives (FP) and False Negatives (FN)

- **False Positives (FP)**: the model *incorrectly* predicts the positive class — the actual label is negative, but the prediction is positive. Also called a *Type I error*.

- **False Negatives (FN)**: the model *incorrectly* predicts the negative class — the actual label is positive, but the prediction is negative. Also called a *Type II error*.

In our example:  
- **FP = 1** — sample 6 is negative (0) but predicted positive (1).  
- **FN = 1** — sample 2 is positive (1) but predicted negative (0).

In [ ]:
FP = np.sum((y_true == 0) & (y_pred == 1))
FN = np.sum((y_true == 1) & (y_pred == 0))

print(f"False Positives (FP): {FP}")
print(f"False Negatives (FN): {FN}")

### 1.3 Derived Rates

From TP, TN, FP, FN we define four rates:

- **True Positive Rate (TPR)** — also called *Sensitivity* or *Recall*: fraction of actual positives correctly identified.  
  $\text{TPR} = \dfrac{TP}{TP + FN}$  
  Example: $\dfrac{3}{3+1} = 0.75$ — the model catches 75% of positive cases.


In [ ]:
TPR = TP / (TP + FN)  # Sensitivity / Recall

print(f"True Positive Rate  (TPR / Recall):      {TPR:.2f}")


- **True Negative Rate (TNR)** — also called *Specificity*: fraction of actual negatives correctly identified.  
  $\text{TNR} = \dfrac{TN}{TN + FP}$  
  Example: $\dfrac{3}{3+1} = 0.75$ — the model correctly dismisses 75% of negative cases.


In [ ]:
TNR = TN / (TN + FP)  # Specificity
print(f"True Negative Rate  (TNR / Specificity):  {TNR:.2f}")


- **False Positive Rate (FPR)** — also called *Fall-out*: fraction of actual negatives incorrectly predicted as positive.  
  $\text{FPR} = \dfrac{FP}{FP + TN} = 1 - \text{TNR}$  
  Example: $\dfrac{1}{1+3} = 0.25$ 


In [ ]:
FPR = FP / (FP + TN)  # Fall-out  (x-axis of the ROC curve)

print(f"False Positive Rate (FPR / Fall-out):     {FPR:.2f}")


- **False Negative Rate (FNR)** — also called *Miss rate*: fraction of actual positives incorrectly predicted as negative.  
  $\text{FNR} = \dfrac{FN}{FN + TP} = 1 - \text{TPR}$  
  Example: $\dfrac{1}{1+3} = 0.25$.


In [ ]:
FNR = FN / (FN + TP)  # Miss rate


print(f"False Negative Rate (FNR / Miss rate):    {FNR:.2f}")

Note: $\text{TPR} + \text{FNR} = 1$ and $\text{TNR} + \text{FPR} = 1$ by definition.

In [ ]:
print()
print(f"TPR + FNR = {TPR + FNR:.0f}  |  TNR + FPR = {TNR + FPR:.0f}")

### 1.4 Classification Scenarios

Depending on the type of data, we could have the following classification scenarios

* **Binary Classification**: Each instance can belong to either of two possible classes, and each instance belongs to **exactly one** class.
* **Multi-Class Classification**: We have $C$ possible classes, where each instance belongs to **exactly one** class.
* **Multi-Label Classification**: We have $C$ possible classes, and each instance can belong to multiple classes (or none). 

| Scenario | Number of classes | Classes per instance |
|---|:---:|:---:|
| Binary Classification | 2 | Exactly 1 |
| Multi-Class Classification | $C \geq 2$ | Exactly 1 |
| Multi-Label Classification | $C \geq 2$ | 0 or more (up to $C$) |

In the dataset for our project we have a set of sound events that can happen simultaneously and in the same audio frame. This turns our problem into a **multi-label classification problem**. One simple was of performing this type of classification is to have one binary classifier per class, working independently.

In this notebook we will cover multi-class classification metrics for the sake of completeness, and to highlight different types of aggregation that result when dealing with multiple classes.

## 2. Binary Classification Metrics

### 2.1 Accuracy

**Accuracy** is the fraction of all predictions the model got right.

$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$

In our example: $\dfrac{3 + 3}{3 + 3 + 1 + 1} = \dfrac{6}{8} = 0.75$

> **Limitation:** accuracy is misleading on imbalanced datasets — a model that always predicts the majority class can score high while being useless.

In [ ]:
correct = np.sum(y_true == y_pred)
accuracy_manual = correct / len(y_true)
accuracy_sk = accuracy_score(y_true, y_pred)

print(f"Correct predictions: {correct} / {len(y_true)}")
print(f"Accuracy (manual):   {accuracy_manual:.2f}")
print(f"Accuracy (sklearn):  {accuracy_sk:.2f}")

### 2.2 Confusion Matrix

The **Confusion Matrix** summarises predictions by counting how often each true class was assigned each predicted class.

For binary classification it is a 2×2 table:

|  | **Predicted Positive** | **Predicted Negative** |
|--|:----------------------:|:----------------------:|
| **Actual Positive** | True Positive (TP) | False Negative (FN) |
| **Actual Negative** | False Positive (FP) | True Negative (TN) |

In our running example: TP = 3, FN = 1, FP = 1, TN = 3.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    display_labels=["Negative", "Positive"],
    colorbar=False,
    ax=ax,
)
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()

### 2.3 Precision, Recall & F1-score

These metrics focus on the **positive class** and capture different aspects of how well the model finds positive cases.

#### Precision — "How reliable are positive predictions?"
$$\text{Precision} = \frac{TP}{TP + FP}$$
Example: $\dfrac{3}{3+1} = 0.75$ — 75% of predicted positives are actually positive.

In [ ]:
# Calculate precision
precision = precision_score(y_true, y_pred)
print(f"Precision: {precision:.2f}")

#### Recall — "How many positives are found?"
$$\text{Recall} = \frac{TP}{TP + FN}$$
Example: $\dfrac{3}{3+1} = 0.75$ — 75% of actual positives are correctly identified.

In [ ]:
# Calculate recall
recall = recall_score(y_true, y_pred)
print(f"Recall: {recall:.2f}")

#### F1-score — Harmonic mean of Precision and Recall
$$\text{F1} = 2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$
Example: $2 \cdot \dfrac{0.75 \times 0.75}{0.75 + 0.75} = 0.75$

In [ ]:
# Calculate F1-score
f1 = f1_score(y_true, y_pred)
print(f"F1-score: {f1:.2f}")

### Precision-Recall Trade-off

Precision and recall are often in tension — improving one tends to hurt the other.

**High precision, low recall** — the model predicts positive only when very confident.  
Predict $\hat{y} = [0,0,1,0,0,0,0,0]$: catches 1 of 4 positives, 0 false alarms.  
→ Precision = 1.00, Recall = 0.25, F1 = 0.40

**Low precision, high recall** — the model predicts positive for everything.  
Predict $\hat{y} = [1,1,1,1,1,1,1,1]$: catches all positives, but also all negatives.  
→ Precision = 0.50, Recall = 1.00, F1 = 0.67

> F1-score penalises extreme imbalances between precision and recall.

In [ ]:
y_true_pr = np.array([1, 0, 1, 1, 0, 1, 0, 0])  # 4 positives, 4 negatives

# High precision, low recall: predict positive only once (correctly)
y_pred_hp_lr = np.array([0, 0, 1, 0, 0, 0, 0, 0])

# Low precision, high recall: predict everything as positive
y_pred_lp_hr = np.array([1, 1, 1, 1, 1, 1, 1, 1])

for label, yp in [("High P / Low R", y_pred_hp_lr), ("Low P  / High R", y_pred_lp_hr)]:
    p = precision_score(y_true_pr, yp, zero_division=0)
    r = recall_score(y_true_pr, yp)
    f = f1_score(y_true_pr, yp, zero_division=0)
    print(f"{label}:  Precision={p:.2f}  Recall={r:.2f}  F1={f:.2f}")

### 2.4 Balanced Accuracy

**Balanced Accuracy** is the average per-class recall. For binary classification:

$$
\text{Balanced Accuracy} = \frac{\text{TPR} + \text{TNR}}{2}
$$

It gives equal weight to each class, making it reliable when the **dataset is imbalanced**. Plain accuracy would be dominated by the majority class.

In our example: $\dfrac{0.75 + 0.75}{2} = 0.75$ (accuracy and balanced accuracy agree here because the dataset has equal class sizes).

In [ ]:
bal_acc_manual = (TPR + TNR) / 2
bal_acc_sk = balanced_accuracy_score(y_true, y_pred)

print(f"Balanced Accuracy (manual):  {bal_acc_manual:.2f}")
print(f"Balanced Accuracy (sklearn): {bal_acc_sk:.2f}")
print(f"  = (TPR + TNR) / 2 = ({TPR:.2f} + {TNR:.2f}) / 2 = {bal_acc_manual:.2f}")

### Imbalanced Dataset Example

When classes are imbalanced, **plain accuracy is misleading**.

Consider 8 negatives and 2 positives. A model that **always predicts negative**:

| Metric | Value | Interpretation |
|--------|------:|----------------|
| Accuracy | 0.80 | "80 % correct" — looks good! |
| Balanced Accuracy | 0.50 | Equal to random guessing |

Balanced accuracy correctly exposes that this model has learned nothing about the minority class (TPR = 0, TNR = 1).

In [ ]:
# Imbalanced dataset: 8 negatives, 2 positives (80 / 20 split)
y_true_imb = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 1])
y_pred_majority = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])  # always predict negative

acc = accuracy_score(y_true_imb, y_pred_majority)
bal_acc = balanced_accuracy_score(y_true_imb, y_pred_majority)
tpr_imb = recall_score(y_true_imb, y_pred_majority, zero_division=0)
tnr_imb = recall_score(y_true_imb, y_pred_majority, pos_label=0)

print(f"TPR (recall on positives): {tpr_imb:.2f}  — misses all positives")
print(f"TNR (recall on negatives): {tnr_imb:.2f}  — correctly rejects all negatives")
print()
print(f"Accuracy:          {acc:.2f}  ← misleadingly high")
print(
    f"Balanced Accuracy: {bal_acc:.2f}  ← reveals random performance on minority class"
)

### 2.5 ROC AUC

The **ROC curve** (Receiver Operating Characteristic) plots the True Positive Rate against the False Positive Rate as the classification threshold is swept from 1 to 0.

- A **random classifier** follows the diagonal (AUC = 0.5).
- A **perfect classifier** reaches the top-left corner (TPR = 1, FPR = 0), giving AUC = 1.0.

**ROC AUC** is the area under this curve — equal to the probability that the model ranks a random positive higher than a random negative. It is **threshold-independent** and works well when classes are balanced.

In [ ]:
# Predicted probability scores for the positive class (same running example)
y_score = np.array([0.9, 0.2, 0.4, 0.8, 0.1, 0.85, 0.6, 0.15])

fpr_vals, tpr_vals, _ = roc_curve(y_true, y_score)
roc_auc = roc_auc_score(y_true, y_score)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(fpr_vals, tpr_vals, marker="o", label=f"Classifier (AUC = {roc_auc:.2f})")
ax.plot([0, 1], [0, 1], "k--", label="Random (AUC = 0.50)")
ax.set_xlabel("False Positive Rate (FPR)")
ax.set_ylabel("True Positive Rate (TPR)")
ax.set_title("ROC Curve")
ax.legend()
plt.tight_layout()
plt.show()
print(f"ROC AUC: {roc_auc:.2f}")

### 2.6 PR AUC

The **Precision-Recall curve** plots Precision against Recall as the classification threshold is swept from 1 to 0.

- A **perfect classifier** reaches (Recall = 1, Precision = 1).
- A **random classifier** with positive rate $p$ gives a flat baseline at Precision = $p$.

**PR AUC** (*Average Precision*) is especially useful when the **positive class is rare** — unlike ROC AUC it is directly sensitive to class imbalance.

> Use ROC AUC when classes are roughly balanced; prefer PR AUC when positives are scarce.

In [ ]:
precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_score)
pr_auc = average_precision_score(y_true, y_score)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(
    recall_vals, precision_vals, marker="o", label=f"Classifier (AP = {pr_auc:.2f})"
)
baseline = y_true.mean()
ax.axhline(baseline, color="k", linestyle="--", label=f"Random (AP ≈ {baseline:.2f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve")
ax.legend()
plt.tight_layout()
plt.show()
print(f"PR AUC (Average Precision): {pr_auc:.2f}")

## 3. Multi-Class Classification Metrics

> Note that we don't need to focus on multi-class classification metrics for our project. We are just covering here for completeness.

We use the following **running example** throughout this section (3 classes, 3 samples each — a balanced dataset):

| # | True class | Predicted class |
|---|:----------:|:---------------:|
| 0 | 0 | 0 |
| 1 | 0 | 0 |
| 2 | 0 | 1 |
| 3 | 1 | 1 |
| 4 | 1 | 1 |
| 5 | 1 | 2 |
| 6 | 2 | 0 |
| 7 | 2 | 2 |
| 8 | 2 | 2 |

Each class has exactly 2 correct and 1 wrong prediction.

> Note that our project is not **multi-class** but **multi-label** classification! However, the content here is useful to know in general, so we add it here as reference, particularly, since it is important to be careful with implicit decisions taken due to the default parameters in `sklearn`.

In [ ]:
y_true_multi = np.array([0, 0, 0, 1, 1, 1, 2, 2, 2])
y_pred_multi = np.array([0, 0, 1, 1, 1, 2, 0, 2, 2])

print("y_true:", y_true_multi)
print("y_pred:", y_pred_multi)

### 3.1 Accuracy

For multi-class problems accuracy generalises directly:

$$\text{Accuracy} = \frac{\text{Correct predictions}}{\text{Total predictions}}$$

In our example: $\dfrac{6}{9} \approx 0.67$ — 2 correct predictions per class.

In [ ]:
correct_multi = np.sum(y_true_multi == y_pred_multi)
acc_multi = accuracy_score(y_true_multi, y_pred_multi)

print(f"Correct: {correct_multi} / {len(y_true_multi)}")
print(f"Accuracy: {acc_multi:.2f}")

### 3.2 Balanced Accuracy

For multi-class problems, balanced accuracy is the **macro average of per-class recall**:

$$\text{Balanced Accuracy} = \frac{1}{C}\sum_{i=0}^{C-1}\text{Recall}_i$$

In our **balanced** example every class has the same recall (2/3), so balanced accuracy equals plain accuracy.

In [ ]:
acc_multi     = accuracy_score(y_true_multi, y_pred_multi)
bal_acc_multi = balanced_accuracy_score(y_true_multi, y_pred_multi)

for cls in range(3):
    mask = (y_true_multi == cls)
    r = (y_pred_multi[mask] == cls).sum() / mask.sum()
    print(f"  Recall class {cls}: {r:.2f}  ({mask.sum()} samples)")
print()
print(f"Accuracy:          {acc_multi:.2f}")
print(f"Balanced Accuracy: {bal_acc_multi:.2f}  (same — dataset is balanced)")

### Imbalanced Multi-Class Example

Consider **6 class-0, 3 class-1, 1 class-2** samples.  
A model that **always predicts class 0**:

| Metric | Value | Interpretation |
|--------|------:|----------------|
| Accuracy | 0.60 | "60 % correct" — looks acceptable |
| Balanced Accuracy | 0.33 | Equal to random guessing for 3 classes |

Balanced accuracy exposes that the model completely ignores the two minority classes.

In [ ]:
# Imbalanced: 6x class 0, 3x class 1, 1x class 2
y_true_imb_multi  = np.array([0, 0, 0, 0, 0, 0, 1, 1, 1, 2])
y_pred_all0_multi = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])  # always predict class 0

acc_imb     = accuracy_score(y_true_imb_multi, y_pred_all0_multi)
bal_acc_imb = balanced_accuracy_score(y_true_imb_multi, y_pred_all0_multi)

for cls in range(3):
    mask = (y_true_imb_multi == cls)
    r = (y_pred_all0_multi[mask] == cls).sum() / mask.sum()
    print(f"  Recall class {cls}: {r:.2f}  ({mask.sum()} samples)")
print()
print(f"Accuracy:          {acc_imb:.2f}  \u2190 misleadingly high")
print(f"Balanced Accuracy: {bal_acc_imb:.2f}  \u2190 reveals the model ignores minority classes")

### 3.3 Confusion Matrix

For multi-class problems the confusion matrix is a **C×C table**: rows are actual classes, columns are predicted classes. The diagonal holds correctly classified samples; off-diagonal entries reveal specific class confusions.

In our balanced example:

|  | **Pred 0** | **Pred 1** | **Pred 2** |
|--|:----------:|:----------:|:----------:|
| **Act 0** | 2 | 1 | 0 |
| **Act 1** | 0 | 2 | 1 |
| **Act 2** | 1 | 0 | 2 |

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
ConfusionMatrixDisplay.from_predictions(
    y_true_multi, y_pred_multi,
    display_labels=["Class 0", "Class 1", "Class 2"],
    colorbar=False,
    ax=ax,
)
ax.set_title("Confusion Matrix (3 classes)")
plt.tight_layout()
plt.show()

### 3.4 Macro, Micro & Weighted F1-score

In multiclass classification, per-class F1-scores must be combined into a single number. The three standard strategies are **macro**, **micro**, and **weighted** averaging.

#### Macro Averaging — equal weight per class
$$\text{Macro F1} = \frac{1}{C} \sum_{i=1}^{C} \text{F1}_i$$
Every class counts equally regardless of size. Best for **imbalanced datasets**.

#### Micro Averaging — equal weight per instance
$$\text{Micro F1} = \frac{2 \cdot \text{Precision}_\mu \cdot \text{Recall}_\mu}{\text{Precision}_\mu + \text{Recall}_\mu}$$
Aggregates TP, FP, FN globally. Dominated by frequent classes.

#### Weighted Averaging — weight by class support
$$\text{Weighted F1} = \frac{\sum_{i=1}^{C} \text{support}_i \cdot \text{F1}_i}{\sum_{i=1}^{C} \text{support}_i}$$
Accounts for class frequency — a middle ground between macro and micro. (support  is number of actual (true) instances of each class in the dataset).

In [ ]:
# Classification report for the running example
print("Classification Report:")
print(classification_report(y_true_multi, y_pred_multi, target_names=["Class 0", "Class 1", "Class 2"]))

In [ ]:
f1_macro    = f1_score(y_true_multi, y_pred_multi, average='macro')
f1_micro    = f1_score(y_true_multi, y_pred_multi, average='micro')
f1_weighted = f1_score(y_true_multi, y_pred_multi, average='weighted')

print(f"Macro F1-score:    {f1_macro:.2f}  (equal weight per class)")
print(f"Micro F1-score:    {f1_micro:.2f}  (equal weight per instance)")
print(f"Weighted F1-score: {f1_weighted:.2f}  (weighted by class support)")

With our **balanced, symmetric** example all three scores coincide (~0.67) because every class has identical precision, recall, and support.

The three strategies diverge when classes are **imbalanced** or errors are unevenly distributed:

- **Macro** is pulled down by a class the model handles poorly, regardless of its size.
- **Micro** is dominated by frequent classes — a model that ignores a rare class can still score high.
- **Weighted** sits between the two: it accounts for class size but still gives some weight to minority classes.

### Imbalanced F1-score Example

Using an imbalanced dataset (**80 class-0, 15 class-1, 5 class-2**), three classifiers each optimize for a different pattern — revealing how the metric choice matters:

| Classifier | Strategy | Highest metric |
|------------|----------|----------------|
| A | Majority-biased — very good on class 0 | **Micro F1** |
| B | Minority-biased — perfect on classes 1 \& 2 | **Macro F1** |
| C | Conservative — never falsely predicts class 0 | **Weighted F1** |

In [ ]:
y_true_imb = np.array([0]*80 + [1]*15 + [2]*5)

# Clf A: majority-biased — good at class 0, mediocre at minority classes
y_pred_a = np.array([0]*78+[1]*1+[2]*1 + [0]*5+[1]*9+[2]*1 + [0]*3+[1]*1+[2]*1)

# Clf B: minority-biased — perfect on classes 1 & 2, terrible on class 0
y_pred_b = np.array([0]*10+[1]*70 + [1]*15 + [2]*5)

# Clf C: conservative — never misclassifies as class 0, so FP_0 = 0
y_pred_c = np.array([0]*48+[1]*25+[2]*7 + [1]*10+[2]*5 + [1]*2+[2]*3)

print(f"{'':<25} {'Clf A (Micro wins)':>20} {'Clf B (Macro wins)':>20} {'Clf C (Weighted wins)':>22}")
print("-" * 90)
for avg in ['macro', 'micro', 'weighted']:
    scores = [f1_score(y_true_imb, yp, average=avg, zero_division=0)
              for yp in [y_pred_a, y_pred_b, y_pred_c]]
    best = scores.index(max(scores))
    row = f"{avg.capitalize() + ' F1':<25}"
    for j, s in enumerate(scores):
        marker = " ◄" if j == best else "  "
        row += f" {s:>19.3f}{marker}"
    print(row)

### 3.5 Interactive Multi-Class Classification Example

Adjust the confusion-matrix sliders to see how macro, micro, and weighted F1-scores respond to different class distributions and error patterns.

In [ ]:
# Define the number of classes
num_classes = 3
classes = [0, 1, 2]

# Create sliders for each cell in the confusion matrix
sliders = {}
for i in classes:
    for j in classes:
        sliders[f"T{i}P{j}"] = widgets.IntSlider(
            value=0, min=0, max=20, description=f"C{i}{j}"
        )

# Arrange the sliders in a grid
conf_matrix_ui = widgets.GridBox(
    children=[sliders[f"T{i}P{j}"] for i in classes for j in classes],
    layout=widgets.Layout(
        width="100%",
        grid_template_columns="repeat(3, 200px)",
        grid_template_rows="repeat(3, 60px)",
        grid_gap="17px",
    ),
)

# Initialize confusion matrix with some values
sliders["T0P0"].value = 5  # True class 0 predicted as class 0
sliders["T1P1"].value = 4  # True class 1 predicted as class 1
sliders["T2P2"].value = 6  # True class 2 predicted as class 2
sliders["T0P1"].value = 2  # True class 0 predicted as class 1
sliders["T1P2"].value = 3  # True class 1 predicted as class 2
sliders["T2P0"].value = 1  # True class 2 predicted as class 0

Now, let's define a function that will update the metrics based on the confusion matrix values.

In [ ]:
def update_metrics(**kwargs):
    # Build the confusion matrix from the sliders
    cm = np.array(
        [
            [kwargs[f"T0P0"], kwargs[f"T0P1"], kwargs[f"T0P2"]],
            [kwargs[f"T1P0"], kwargs[f"T1P1"], kwargs[f"T1P2"]],
            [kwargs[f"T2P0"], kwargs[f"T2P1"], kwargs[f"T2P2"]],
        ]
    )

    # Display the confusion matrix
    cm_df = pd.DataFrame(
        cm,
        index=[f"Actual {i}" for i in classes],
        columns=[f"Predicted {i}" for i in classes],
    )
    print("Confusion Matrix:")
    display(cm_df)

    # Flatten the confusion matrix to get true labels and predicted labels
    y_true = []
    y_pred = []
    for i in classes:
        for j in classes:
            count = cm[i][j]
            y_true.extend([i] * count)
            y_pred.extend([j] * count)

    if len(y_true) == 0:
        print("No samples to evaluate.")
        return

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

    precision_micro = precision_score(y_true, y_pred, average="micro", zero_division=0)
    recall_micro = recall_score(y_true, y_pred, average="micro", zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=0)

    # Display metrics
    print(f"Accuracy: {accuracy:.2f}")
    print("\nMacro-Averaged Metrics:")
    print(f"Precision (Macro): {precision_macro:.2f}")
    print(f"Recall (Macro): {recall_macro:.2f}")
    print(f"F1-score (Macro): {f1_macro:.2f}")

    print("\nMicro-Averaged Metrics:")
    print(f"Precision (Micro): {precision_micro:.2f}")
    print(f"Recall (Micro): {recall_micro:.2f}")
    print(f"F1-score (Micro): {f1_micro:.2f}")

    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

Now, we can set up the interactive display.

In [ ]:
out = widgets.interactive_output(update_metrics, sliders)
display(conf_matrix_ui, out)

- Each slider represents the count of instances where a true class is predicted as a certain class.
- For example, `T0P1` represents the number of instances where the true class is 0 but predicted as 1.
- Adjust the sliders to change the confusion matrix and observe how the macro and micro F1-scores change.

- **Macro-Averaged Metrics**: Calculate the metric independently for each class and then take the average (treat all classes equally).
- **Micro-Averaged Metrics**: Aggregate the contributions of all classes to compute the average metric (treat all instances equally).

